# Ingest results.json file Assignment


## Assignment
- create a DF loading the constructors.json file in the raw container
- rename columns to (result_id,race_id,driver_id,constructor_id,position_text,position_order,fastest_lap,fastest_lap_time,fastest_lap_speed)
- add a new columns ingestion_timestampt
- save the file in parquet file in the processed container.
- drop statusId
- verify the schema of the partquet file

In [0]:
%run "../includes/common_functions"

In [0]:
%run "../includes/configuration"

In [0]:
dbutils.widgets.text("p_date_source","")
#dbutils.widgets.dropdown("p_date_source","Testing",["Testing","Production"])
v_data_source=dbutils.widgets.get("p_date_source")

In [0]:
from pyspark.sql.functions import current_timestamp,col


In [0]:
display(spark.read.json(f"{raw_folder_path}/results.json").summary())





In [0]:
results_schema="resultId INT,raceId INT, driverId INT, constructorId INT,number STRING, grid INT, position STRING, positionText STRING, positionOrder INT, points DOUBLE, laps INT, time STRING, milliseconds INT,fastestLap STRING, rank INT, fastestLapSpeed STRING, fastestLapTime STRING, statusId INT"

In [0]:
results_df = spark.read.schema(results_schema).json(f"{raw_folder_path}/results.json")


In [0]:
results_df=results_df.select(col("resultId").alias("result_id"),col("raceId").alias("race_id"),col("driverId").alias("driver_id"),col("constructorId").alias("constructor_id"),col("number"),col("grid"),col("position"),col("positionText").alias("position_text"),col("positionOrder").alias("position_order"),col("points"),col("laps"),col("time"),col("milliseconds"),col("fastestLap").alias("fastest_lap"),col("rank"),col("fastestLapSpeed").alias("fastest_lap_speed"),col("fastestLapTime").alias("fastest_lap_time"))


In [0]:
results_df=add_ingestion_timestamp(results_df)
results_df = add_data_source(results_df,v_data_source)

In [0]:
#display(results_df)

## Write DF into parquet file

In [0]:
#results_df.write.mode("overwrite").partitionBy("race_id").parquet(f"{processed_folder_path}/results")

In [0]:
results_df.write.mode("overwrite").partitionBy("race_id").format("parquet").saveAsTable("f1_processed_db.results")

In [0]:
#df=spark.read.parquet(f"{processed_folder_path}/results")
#df.printSchema()

In [0]:
dbutils.notebook.exit("Success")